In [ ]:
%%capture

%pip install langchain-community -U
%pip install langchain-google-genai
%pip install pypdf
%pip install langchain
%pip install langchain-chroma
%pip install langchain -U

In [ ]:
# Imports
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

In [ ]:
# Carregar e extrair pdf
loader = PyPDFLoader('os-sertoes.pdf')
documents = loader.load()

# Criar chuncks
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=20)
chunks = text_splitter.split_documents(documents)

In [ ]:
# Embeddings
embeddings_model = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

# Extrair o conteúdo de cada Document para gerar embeddings
text_contents = [doc.page_content for doc in chunks]
embeddings = embeddings_model.embed_documents(text_contents)

In [ ]:
# Criação do retriever para encontrar os chunks mais relevantes 
vectorstore = Chroma.from_documents(chunks, embedding=embeddings_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
# Init modelo Gemini e criação do prompt template
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.2)
prompt = ChatPromptTemplate.from_template("""
    Você é um bibliotecário. Responda as perguntas baseadas no contexto fornecido.
                                          
    Context: {context}
                                          
    Pergunta: {input}
""")

document_chain = create_stuff_documents_chain(llm, prompt)

In [ ]:
asks = [
    "Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?",
    "Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas características com o ambiente em que vivem?",
    "Qual foi o contexto histórico e político que levou à Guerra de Canudos, segundo Euclides da Cunha?",
    "Como Euclides da Cunha descreve a figura de Antônio Conselheiro e seu papel na Guerra de Canudos?",
    "Quais são os principais aspectos da crítica social e política presentes em \"Os Sertões\"? Como esses aspectos refletem a visão do autor sobre o Brasil da época?"
]

for ask in asks:
    # Recupera os chunks mais relevantes baseado na pergunta
    context = retriever.invoke(ask)

    # Invoke da LLM com a pergunta e os documentos obtidos no RAG.
    response = document_chain.invoke({"input": ask, "context": context})

    # Imprima a resposta
    print(f"Pergunta: {ask}\nResposta: {response}\n\n")

Pergunta: Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?
Resposta: Euclides da Cunha retrata o sertão nordestino em *Os Sertões* como um ambiente hostil e marcado por um ciclo vicioso de fatores naturais interligados.  Ele descreve um clima extremo, com secas prolongadas e chuvas torrenciais que causam erosão e degradação do solo.  A vegetação é esparsa e a paisagem é árida e desolada, com relevos acidentados e solos pouco férteis.  Essa combinação de fatores resulta em um ambiente que dificulta a agricultura e a sobrevivência, levando à pobreza e à miséria da população.

Para Cunha, o martírio do homem no sertão é um reflexo do "martírio secular da Terra". A natureza inclemente não apenas impõe dificuldades físicas, como a sede, mas também condiciona a economia e a vida social, criando um ciclo de pobreza e sofrimento que se perpetua ao longo das gerações.  A falta de recursos hídricos, apesar de iniciativ